# 🚀 Phase 2: Simple Baseline Model training (ViT + GPT-2)
### Multimodal Medical Report Generator

This notebook demonstrates how to construct, train, and evaluate a baseline Vision-Language Model (VLM) on a subset of the **IU X-Ray dataset** using Google Colab. We use `google/vit-base-patch16-224` as the frozen vision encoder and `gpt2` (124M parameters) as our language decoder, connected via a projection layer.

## 1. Environment Setup
If running on Google Colab, uncomment and run the following cell to install necessary libraries:

In [ ]:
# !pip install -r ../requirements.txt

In [ ]:
import os
import sys
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from tqdm import tqdm

# Ensure local src imports work
sys.path.append(os.path.abspath(".."))
from src.data.dataset import MultimodalMedicalDataset
from src.models.multimodal import MultimodalMedicalReportGenerator
from src.data.preprocess import run_pipeline

## 2. Load Data & Preprocessing
First, we make sure that our preprocessed splits exist. If they don't, we can download and parse the dataset. 

*Note: Programmatic download of images (`NLMCXR_png.tgz`) might take a few minutes as it is ~1-2GB. If you just want to run a dry-run of the code parsing, you can run preprocess with `--skip-images`.*

In [ ]:
csv_path = "../data/splits/train.csv"
img_dir = "../data/raw/images"

if not os.path.exists(csv_path):
    print("Preprocessed data not found. Running the download/preprocess pipeline...")
    # To download everything including images, leave download_images=True
    run_pipeline(download_images=True)
else:
    print("Preprocessed split files detected!")

## 3. Configure Tokenizer and Dataset
We load the tokenizer and define our PyTorch `MultimodalMedicalDataset`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Instantiate train & val datasets
train_dataset = MultimodalMedicalDataset(
    csv_file="../data/splits/train.csv",
    img_dir=img_dir,
    tokenizer=tokenizer,
    image_size=224,
    split="train",
    max_length=64
)

val_dataset = MultimodalMedicalDataset(
    csv_file="../data/splits/val.csv",
    img_dir=img_dir,
    tokenizer=tokenizer,
    image_size=224,
    split="val",
    max_length=64
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

print(f"Batches in train loader: {len(train_loader)}")
print(f"Batches in val loader:   {len(val_loader)}")

## 4. Instantiate Baseline VLM Model
We create a model wrapping `google/vit-base-patch16-224` (frozen) and `gpt2` using a simple MLP projection layer.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running model on device: {device}")

model = MultimodalMedicalReportGenerator(
    vision_model="google/vit-base-patch16-224",
    language_model="gpt2",
    projector_type="mlp",
    freeze_vision=True,
    use_lora=False  # Keep LLM frozen for Stage 1 projector alignment
)
model = model.to(device)

## 5. Simple Training Loop
Let's execute a quick 2-epoch alignment training loop to check forward pass, backward propagation, and loss updates.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
epochs = 2

print("Starting baseline training...")
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for batch in progress_bar:
        images = batch["image"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Clear gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images, input_ids, attention_mask, labels)
        loss = outputs["loss"]
        
        # Backward pass & step
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
        
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

## 6. Autoregressive Generation Inference
Let's pass a chest X-ray image and have our model generate a radiology report findings statement!

In [ ]:
model.eval()
# Fetch a single validation sample
sample_batch = next(iter(val_loader))
sample_image = sample_batch["image"][:1].to(device)  # Batch size 1

# Generate report
generated_reports = model.generate(
    images=sample_image,
    tokenizer=tokenizer,
    max_new_tokens=40,
    temperature=0.7,
    prompt_text="Findings:"
)

print("\n--- Generated Report Findings ---")
print(generated_reports[0])

print("\n--- Ground Truth Report ---")
gt_ids = sample_batch["input_ids"][0]
print(tokenizer.decode(gt_ids, skip_special_tokens=True))